[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day3_Remote_Sensing_Fundamentals_and_Indices/01_long_term_surface_water_jrc.ipynb)

# Day 3 - Fundamentals of Remote Sensing & Spectral Indices
## Session 1: Long Term Surface Water JRC

| | |
|---|---|
| **Fecha / Date** | 2026-10-14 (Wednesday / Miercoles) |
| **Horario / Time** | 09:00 - 10:30 |
| **Entidad / Entity** | UoM |
| **Instructores / Instructors** | Sebastian, Crhisthian |

### Contenido / Content

> Long-Term Surface Water Dynamics using global products. Practical exercise with JRC Global Surface Water Explorer

### Objetivos de aprendizaje / Learning objectives

- [ ] Explain how the JRC Global Surface Water product is built, what each of its four
      datasets contains, and what the pixel values mean
- [ ] Display and read the mapping layers - occurrence, seasonality, recurrence, transition
      and change - over the Titicaca-Desaguadero study area
- [ ] Convert a water classification into a **number**: surface water area in km2, and
      check how much of the area was actually observed
- [ ] Build a monthly water-area time series for a lake and read its seasonal cycle and
      its interannual trend
- [ ] Continue the series past the end of the JRC product using Sentinel-2 and MNDWI, and
      state why the two curves do not coincide exactly

### Datos necesarios / Required data

No local files are needed. Everything is read from the Earth Engine catalogue:

| Dataset | Earth Engine ID | Period |
|---|---|---|
| Mapping layers | `JRC/GSW1_4/GlobalSurfaceWater` | 1984-2021 (one image) |
| Monthly history | `JRC/GSW1_4/MonthlyHistory` | 1984-03 to 2021-12 |
| Yearly history | `JRC/GSW1_4/YearlyHistory` | 1984 to 2021 |
| Sentinel-2 L2A | `COPERNICUS/S2_SR_HARMONIZED` | 2017 to present |

The Earth Engine account created in Day 2, session 4 is required.

---

## 1.1 - Teoria / Theory: what the JRC Global Surface Water product is

### The problem it solves

Mapping water on a single satellite image is easy - water absorbs the near and short-wave
infrared, so it is dark in those bands and an index such as NDWI or MNDWI separates it in
one line of code. The hard part is the **time dimension**. A lake photographed in April is
not the same lake in October, and a single image cannot tell you whether a flooded pixel is
a permanent lake, a seasonal wetland, an irrigation reservoir or a one-off flood.

The **JRC Global Surface Water** dataset (Pekel et al., *Nature*, 2016) answers that
question globally. The team classified **every Landsat 5, 7 and 8 pixel ever acquired** -
4,716,475 scenes, about 1.8 million billion pixels - as water, land or no-data, at 30 m,
from March 1984 to December 2021. The result is not one map but a **record of what each
pixel on Earth did over 38 years**.

Because the classification is already done, you do not compute any index here. You read a
finished product. This is the difference between *processing imagery* and *using a global
product*, and it is the point of this session.

### How the classification was made

An expert system, not a spectral threshold. For each pixel and each date it combines:

1. **Spectral evidence** - the position of the pixel in a multi-band space, using visual and
   infrared bands plus HSV transforms, with decision rules derived from a very large
   training set.
2. **Contextual evidence** - what the neighbouring pixels are doing, which removes isolated
   false positives such as terrain shadow.
3. **Temporal evidence** - what that same pixel did on the previous and following dates.

Every pixel-date also gets a **validity flag**. Clouds, cloud shadow, terrain shadow, snow
and sensor problems (including the Landsat 7 SLC-off stripes after 2003) are labelled
**no-data**, not "land". This distinction matters enormously and we come back to it in
section 1.4: a month with no data is not a month without water.

Validation against 40,124 reference points gives less than 1% false water detections and
less than 5% missed water.

### The four datasets in Earth Engine

| Dataset | What it is | Use it for |
|---|---|---|
| `JRC/GSW1_4/GlobalSurfaceWater` | **One image**, 7 bands, summarising the whole 38-year record | Where water is, how often, and how it changed |
| `JRC/GSW1_4/MonthlyHistory` | **454 images**, one per month, 1984-03 to 2021-12 | Seasonal cycle, month-by-month time series |
| `JRC/GSW1_4/YearlyHistory` | **38 images**, one per year | Permanent vs seasonal water, year by year |
| `JRC/GSW1_4/MonthlyRecurrence` | **12 images**, one per calendar month | Average behaviour of each month of the year |

### The bands of the mapping-layers image

| Band | Meaning | Range |
|---|---|---|
| `occurrence` | % of valid observations in which the pixel was water, 1984-2021 | 0-100 |
| `change_abs` | change in occurrence between 1984-1999 and 2000-2021, in percentage points | -100 to 100 |
| `change_norm` | the same change expressed relative to the earlier occurrence | -100 to 100 |
| `seasonality` | number of months of 2021 in which the pixel was water | 0-12 |
| `recurrence` | % of years in which water returned at least once | 0-100 |
| `transition` | class describing how the pixel changed between the first and last epoch | 0-10 |
| `max_extent` | 1 wherever water was ever detected | 0 or 1 |

Two bands are easy to confuse:

- **`occurrence`** is a frequency *in time overall*. A pixel at 50% was water in half of all
  valid observations - but that could be six months every year, or 19 years flooded followed
  by 19 years dry.
- **`recurrence`** is a frequency *between years*. A pixel at 100% had water in every single
  year, even if only for a few weeks. Seasonal wetlands have **low occurrence and high
  recurrence**; a reservoir built in 2005 has moderate occurrence and low recurrence.

### What this product cannot do

Be explicit about this with any result you produce from it:

- **It measures area, not volume.** Surface extent is not water level and not storage. A
  shallow lake such as Lago Menor loses a lot of area for a small drop in level; a deep lake
  such as Lago Grande barely changes shape while losing metres. Volume needs altimetry
  (session 3) or bathymetry.
- **It stops in December 2021.** Anything after that has to be produced by you - section 1.6.
- **30 m pixels.** Rivers narrower than about 60 m, small canals and small ponds are missed
  or heavily underestimated. Irrigation canals in the altiplano are invisible at this
  resolution.
- **Optical only.** Persistent cloud creates no-data, which is why the wet season has
  systematically fewer valid observations than the dry season - a bias you must check
  before interpreting any seasonal cycle.
- **Landsat 7 SLC-off.** Between 2003 and 2013 a large share of observations comes from a
  sensor with data gaps, so no-data is more frequent in that decade.

---

## 1.2 - Setup: Earth Engine and the study area

We connect to Earth Engine and define the study area as a fixed rectangle covering the
**Titicaca-Desaguadero** system: Lago Grande (Chucuito), Lago Menor (Wi&ntilde;aymarka) and the
upper Desaguadero river. The rectangle is fixed in code so that every participant produces
the same maps and the numbers can be compared across the room.

In [ ]:
# ---------------------------------------------------------------------------
# Libraries (uncomment the pip line on a fresh runtime, e.g. Google Colab)
# ---------------------------------------------------------------------------
#%pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

print("ee", ee.__version__, "| geemap", geemap.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Authenticate and initialise Earth Engine
# ---------------------------------------------------------------------------
# >>> REPLACE with your own Cloud project id <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine connected - project:", EE_PROJECT)

### The study area

`ee.Geometry.Rectangle` takes the corners in the order **[west, south, east, north]**.
The third argument, `False`, makes the edges follow straight lines in latitude/longitude
instead of great circles - the same as the `geodesic: false` option in the Code Editor.

In [ ]:
# ---------------------------------------------------------------------------
# Study area: Titicaca - Desaguadero (same rectangle for everyone)
# ---------------------------------------------------------------------------
ROI = ee.Geometry.Rectangle(
    [-70.8017, -16.8074, -68.1623, -15.0219],   # west, south, east, north
    None, False
)

CENTER_LAT, CENTER_LON = -15.95, -69.45   # centre used for all the maps

print("ROI area: {:,.0f} km2".format(ROI.area(maxError=1000).divide(1e6).getInfo()))

---

## 1.3 - The mapping layers: where the water is and how it behaves

The whole 38-year record is summarised in a **single image**. Loading it is one line; the
rest of this section is about reading each band correctly.

Note `.clip(ROI)`: it restricts the display to our rectangle. Clipping is a display and
computation convenience, not a download - the image still lives on the server.

In [ ]:
# ---------------------------------------------------------------------------
# The mapping-layers image: one image, seven bands, 1984-2021
# ---------------------------------------------------------------------------
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(ROI)

print("Bands:", gsw.bandNames().getInfo())

### `occurrence` - how often each pixel was water

The first and most used band. Read it as a gradient, not as a mask:

- **near 100 (dark blue)** - the permanent lake body
- **intermediate values (light blue, cyan)** - the shoreline that advances and retreats, the
  seasonal margins of Lago Menor, the floodplain of the Desaguadero
- **near 0 / transparent** - land, or water seen only once or twice

The interesting information is in the *intermediate* values. A pixel at 40% is a pixel that
is sometimes lake and sometimes dry ground - which is exactly the part of the system that
matters for irrigation, for grazing on the shoreline *totorales*, and for anyone who built
something close to the water.

`updateMask(gsw.select("occurrence").gt(0))` makes the zero pixels transparent, so the
satellite basemap shows through where water was never detected.

In [ ]:
# ---------------------------------------------------------------------------
# occurrence: % of valid observations in which the pixel was water (0-100)
# ---------------------------------------------------------------------------
occurrence = gsw.select("occurrence")

vis_occurrence = {
    "min": 0, "max": 100,
    "palette": ["#f4e7c5", "#a8d5e2", "#4a9fd1", "#1f5fa0", "#0a2f5e"],
}

Map1 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=8)
Map1.add_basemap("SATELLITE")

# Pixels with occurrence = 0 are made transparent so the basemap shows through
Map1.addLayer(occurrence.updateMask(occurrence.gt(0)), vis_occurrence, "Water occurrence (%)")
Map1.addLayer(ROI, {"color": "yellow"}, "ROI", True, 0.4)

Map1.add_colorbar(vis_occurrence, label="Water occurrence 1984-2021 (%)")
Map1

### `seasonality` and `recurrence` - two different questions

Put side by side, these two bands separate **permanent** water from **seasonal but reliable**
water, which is the distinction a water manager actually needs.

- `seasonality` counts **how many months of the year** the pixel holds water (0-12).
- `recurrence` counts **in what percentage of years** water came back at least once (0-100).

Look at the southern shore of Lago Menor and at the Desaguadero floodplain: low seasonality
combined with high recurrence is a wetland that floods every year for a short period. That
is a very different management object from a permanent lake, even though both are "water"
on a single-date map.

`geemap.linked_maps` shows them with synchronised pan and zoom, so anything that differs on
screen comes from the band and not from the view.

In [ ]:
# ---------------------------------------------------------------------------
# seasonality (months per year) and recurrence (% of years)
# ---------------------------------------------------------------------------
seasonality = gsw.select("seasonality")
recurrence = gsw.select("recurrence")

vis_seasonality = {"min": 0, "max": 12,
                   "palette": ["#ffffff", "#cfe8c8", "#7fc97f", "#3690c0", "#0b3d7a"]}
vis_recurrence = {"min": 0, "max": 100,
                  "palette": ["#ffffff", "#fde0a0", "#f4a261", "#4a9fd1", "#0a2f5e"]}

geemap.linked_maps(
    rows=1, cols=2, height="450px",
    center=[CENTER_LAT, CENTER_LON], zoom=8,
    ee_objects=[seasonality.updateMask(seasonality.gt(0)),
                recurrence.updateMask(recurrence.gt(0))],
    vis_params=[vis_seasonality, vis_recurrence],
    labels=["seasonality (months with water, 0-12)",
            "recurrence (% of years with water)"],
    label_position="topright",
)

### `transition` - what changed between the first and the last epoch

A categorical band comparing the period **1984-1999** with **2000-2021**. Each pixel is
placed in one of eleven classes. The four that usually carry the story are:

| Code | Class | Meaning |
|---|---|---|
| 1 | Permanent | water in both epochs - the stable lake body |
| 2 | New permanent | was not water, is now permanently water |
| 3 | **Lost permanent** | was permanent water, is not water any more |
| 6 | **Lost seasonal** | was seasonal water, is not water any more |

Classes 3 and 6 are the ones to look for on the shallow margins. Use the layer control to
turn the transition layer on and off over the satellite basemap, and check whether what is
marked as "lost" corresponds to a real dried surface or to a shoreline that simply moved.

In [ ]:
# ---------------------------------------------------------------------------
# transition: 11 classes comparing 1984-1999 with 2000-2021
# ---------------------------------------------------------------------------
transition = gsw.select("transition")

# The order of the palette follows the class codes 0, 1, 2, ... 10
TRANSITION_CLASSES = [
    "0 No change", "1 Permanent", "2 New permanent", "3 Lost permanent",
    "4 Seasonal", "5 New seasonal", "6 Lost seasonal", "7 Seasonal to permanent",
    "8 Permanent to seasonal", "9 Ephemeral permanent", "10 Ephemeral seasonal",
]
TRANSITION_PALETTE = [
    "#ffffff", "#0000ff", "#22b14c", "#d1102d", "#99d9ea", "#b5e61d",
    "#e6a1aa", "#ff7f27", "#ffc90e", "#7f7f7f", "#c3c3c3",
]

vis_transition = {"min": 0, "max": 10, "palette": TRANSITION_PALETTE}

Map2 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=8)
Map2.add_basemap("SATELLITE")

# Class 0 ("No change") is masked out so that only the pixels that changed are drawn
Map2.addLayer(transition.updateMask(transition.gt(0)), vis_transition, "Transition class")
Map2.add_legend(title="Transition class",
                legend_dict=dict(zip(TRANSITION_CLASSES, TRANSITION_PALETTE)))

Map2

### `change_abs` - the gain or loss of occurrence, in percentage points

The same comparison as `transition`, but continuous instead of categorical. A value of -40
means the pixel was water 40 percentage points *less often* in 2000-2021 than in 1984-1999.

Red is loss, green is gain, and the diverging palette is centred on zero so that the two
directions are visually symmetric.

Read the result with care. A shoreline that moved outwards produces a band of red on one
side and a band of green on the other; that is a **redistribution**, not a net loss. To know
whether the system lost water you need the total area, which is what the next section
computes.

In [ ]:
# ---------------------------------------------------------------------------
# change_abs: occurrence in 2000-2021 minus occurrence in 1984-1999 (pp)
# ---------------------------------------------------------------------------
change = gsw.select("change_abs")

# Diverging palette: red = loss of occurrence, white = no change, green = gain
vis_change = {"min": -50, "max": 50,
              "palette": ["#b2182b", "#ef8a62", "#f7f7f7", "#7fbf7b", "#1b7837"]}

Map3 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=8)
Map3.add_basemap("SATELLITE")
Map3.addLayer(change.updateMask(gsw.select("max_extent").eq(1)), vis_change,
              "Change in occurrence (pp)")
Map3.add_colorbar(vis_change, label="change_abs: loss (-) / gain (+), percentage points")

Map3

---

## 1.4 - From pixels to numbers: measuring water area

A map shows *where*. To answer *how much*, and later *when*, we need a number. The recipe is
always the same and it is worth memorising, because it works for any binary mask:

1. **Make a 0/1 mask** of the class you want - `eq()`, `gt()`, etc.
2. **Multiply it by `ee.Image.pixelArea()`**, an image whose value is the true area of each
   pixel in m2. Pixels of a lat/lon grid are not all the same size, so never just count
   pixels and multiply by 900.
3. **Sum it over the region** with `reduceRegion(ee.Reducer.sum(), ...)`.
4. **Divide by 1e6** to get km2.

Two arguments of `reduceRegion` decide whether the cell runs in three seconds or times out:

- **`scale`** - the pixel size used for the computation, in metres. `30` is the native
  resolution. Coarser values are faster; because Earth Engine averages the underlying
  pixels when it builds the coarser level, the area estimate stays close, it is the fine
  detail of the shoreline that is lost.
- **`maxPixels`** - a safety limit. Raise it when Earth Engine complains, but only after
  checking that the region and the scale are the ones you intended.

In [ ]:
# ---------------------------------------------------------------------------
# Permanent water (occurrence >= 90%) and maximum extent, over the whole ROI
# ---------------------------------------------------------------------------
pixel_area = ee.Image.pixelArea()

permanent = occurrence.gte(90).multiply(pixel_area).rename("permanent_m2")
max_ext = gsw.select("max_extent").eq(1).multiply(pixel_area).rename("max_extent_m2")

areas = permanent.addBands(max_ext).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=ROI,
    scale=100,          # coarser than 30 m: the ROI is very large
    maxPixels=1e10,
)

result = areas.getInfo()
perm_km2 = result["permanent_m2"] / 1e6
maxe_km2 = result["max_extent_m2"] / 1e6

print("Permanent water  (occurrence >= 90%) : {:8,.0f} km2".format(perm_km2))
print("Maximum extent   (ever seen as water): {:8,.0f} km2".format(maxe_km2))
print("Variable margin  (difference)        : {:8,.0f} km2".format(maxe_km2 - perm_km2))

The third number is the interesting one. The difference between the maximum extent and the
permanent core is the **surface that is sometimes water and sometimes not** - the part of
the system that moves. Compare it with the permanent area: a large ratio means a shallow,
flat-bottomed system where small changes in level move the shoreline a long way.

> **Check before you believe it.** Run the cell again with `scale=30` on a small polygon and
> with `scale=500` on the same polygon. If the two numbers differ by more than a few percent,
> the shoreline is more complex than the coarse scale can represent and you should report the
> finer one.

---

## 1.5 - Application: the water-area time series of a lake

This is the core of the session. We take a lake, reduce **every monthly image** of the JRC
record to a single number - the water area in km2 - and plot the result.

### Step 1 - Choose the water body

Draw a **rectangle or polygon** on the map below, around a single water body: Lago Menor,
a bay of Lago Grande, a stretch of the Desaguadero, a reservoir. Two rules:

- **One water body per polygon.** A polygon containing Lago Menor *and* part of Lago Grande
  adds the two areas and the seasonal signal of the shallow one is buried by the constant
  area of the deep one.
- **Leave a margin of dry land** around the water body. The polygon must contain the whole
  shoreline at its maximum extent, otherwise the series is truncated at the edge of the box
  and a flood looks like a plateau.

If you draw nothing, the notebook falls back to **Lago Menor (Wi&ntilde;aymarka)**, the shallow
southern basin of Titicaca, so the notebook runs end to end.

The occurrence layer is displayed to help you place the boundaries.

In [ ]:
# ---------------------------------------------------------------------------
# Draw here the water body to be monitored
# ---------------------------------------------------------------------------
Map_draw = geemap.Map(center=[-16.35, -68.90], zoom=10)   # centred on Lago Menor
Map_draw.add_basemap("SATELLITE")
Map_draw.addLayer(occurrence.updateMask(occurrence.gt(0)), vis_occurrence, "Water occurrence (%)")

Map_draw

In [ ]:
# ---------------------------------------------------------------------------
# The region: your drawn polygon, or Lago Menor if nothing was drawn
# ---------------------------------------------------------------------------
LAGO_MENOR = ee.Geometry.Rectangle([-69.15, -16.65, -68.55, -16.10], None, False)

region = Map_draw.user_roi if Map_draw.user_roi is not None else LAGO_MENOR
region_km2 = region.area(maxError=100).divide(1e6).getInfo()

print("Using:", "your drawn polygon" if Map_draw.user_roi is not None else "Lago Menor (default)")
print("Region area: {:,.0f} km2".format(region_km2))

### Step 2 - Filter the monthly collection

`JRC/GSW1_4/MonthlyHistory` has one image per month with a single band, `water`:

| Value | Meaning |
|---|---|
| 0 | **no data** - cloud, shadow, snow, sensor gap, or no acquisition |
| 1 | not water |
| 2 | water |

We take the **last ten years of the product, 2012 to 2021**. The product ends in December
2021; section 1.6 shows how to continue from there.

In [ ]:
# ---------------------------------------------------------------------------
# Monthly water history, last 10 years of the JRC product
# ---------------------------------------------------------------------------
START = "2012-01-01"
END   = "2022-01-01"

monthly = (ee.ImageCollection("JRC/GSW1_4/MonthlyHistory")
             .filterDate(START, END)
             .filterBounds(region))

print("Period :", START, "to", END)
print("Months :", monthly.size().getInfo())

### Step 3 - Reduce every month to two numbers

For each month we compute **two** areas, not one:

- `water_km2` - the area classified as water (value 2).
- `valid_km2` - the area that was actually **observed**, that is, everything that is not
  no-data (value different from 0).

The second one is the quality control. If a month was 70% cloud, `water_km2` is measured on
the remaining 30% and is meaningless as a lake area - but it still plots as a point on the
chart and it still looks like a drought. Computing `valid_km2` is what lets us throw those
months away instead of interpreting noise.

`SCALE` is set to 60 m rather than 30 m because we are reducing 120 images in one request.
If the cell times out, raise it to 100, or draw a smaller polygon.

In [ ]:
# ---------------------------------------------------------------------------
# One function applied to every monthly image: water area + observed area
# ---------------------------------------------------------------------------
SCALE = 60          # metres; raise to 100 if the cell times out

def monthly_area(image):
    water_mask = image.select("water").eq(2)      # 2 = water
    valid_mask = image.select("water").neq(0)     # 0 = no data

    water_m2 = water_mask.multiply(pixel_area).rename("water_m2")
    valid_m2 = valid_mask.multiply(pixel_area).rename("valid_m2")

    stats = water_m2.addBands(valid_m2).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, scale=SCALE, maxPixels=1e10)

    return ee.Feature(None, {
        "date": image.date().format("YYYY-MM-dd"),
        "water_km2": ee.Number(stats.get("water_m2")).divide(1e6),
        "valid_km2": ee.Number(stats.get("valid_m2")).divide(1e6),
    })

series = monthly.map(monthly_area)

# getInfo() is where the computation actually happens - this may take 1-2 minutes
rows = [f["properties"] for f in series.getInfo()["features"]]
print("Months returned:", len(rows))

In [ ]:
# ---------------------------------------------------------------------------
# Into a pandas table, and flag the months with too little valid data
# ---------------------------------------------------------------------------
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

# What fraction of the polygon was actually observed that month
df["coverage"] = df["valid_km2"] / region_km2

MIN_COVERAGE = 0.80          # keep months where at least 80% of the polygon was observed
df["usable"] = df["coverage"] >= MIN_COVERAGE

print("Total months        :", len(df))
print("Usable months       :", int(df["usable"].sum()))
print("Discarded (coverage):", int((~df["usable"]).sum()))
df.head()

### Step 4 - Plot the series

Two panels, sharing the same time axis:

- **Top** - the water area. Points that failed the coverage test are drawn hollow and grey,
  and the line skips them, so a gap in the line means "we do not know", not "no water".
- **Bottom** - the coverage itself, with the 80% threshold marked. This panel is not
  decoration: it is the evidence that the top panel can be trusted, and it usually shows the
  wet season (December to March) losing observations to cloud.

A dashed horizontal line marks the mean area of the usable months, so departures are read
against the period average rather than against zero.

In [ ]:
# ---------------------------------------------------------------------------
# Water area and data coverage, 2012-2021
# ---------------------------------------------------------------------------
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d9d8d2"
BLUE, GREY, RED = "#1f6aa5", "#b8b6ae", "#c0392b"

good = df[df["usable"]]
bad = df[~df["usable"]]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1], "hspace": 0.18})

# --- top: water area -------------------------------------------------------
ax1.plot(good.index, good["water_km2"], color=BLUE, linewidth=2,
         marker="o", markersize=4, markeredgecolor="white", markeredgewidth=0.8,
         label="water area (usable months)", zorder=3)
ax1.scatter(bad.index, bad["water_km2"], facecolors="none", edgecolors=GREY,
            s=28, linewidths=1.2, label="discarded (coverage < 80%)", zorder=2)
ax1.axhline(good["water_km2"].mean(), color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax1.set_ylabel("Water area (km2)", fontsize=10, color=MUTED)
ax1.set_title("Surface water area, JRC monthly history", fontsize=11, color=INK, loc="left", pad=6)
ax1.legend(frameon=False, loc="lower left", fontsize=9, labelcolor=INK)

# --- bottom: data coverage -------------------------------------------------
ax2.fill_between(df.index, df["coverage"] * 100, color=BLUE, alpha=0.25, zorder=2)
ax2.plot(df.index, df["coverage"] * 100, color=BLUE, linewidth=1, zorder=3)
ax2.axhline(MIN_COVERAGE * 100, color=RED, linewidth=1, linestyle="--", zorder=4)
ax2.set_ylim(0, 105)
ax2.set_ylabel("Observed (%)", fontsize=10, color=MUTED)
ax2.set_title("Share of the polygon with valid observations", fontsize=10, color=INK,
              loc="left", pad=6)

for ax in (ax1, ax2):
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)

fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.07)
plt.show()

### Step 5 - The average seasonal cycle

The series above mixes two signals: the **seasonal cycle**, which repeats every year, and the
**interannual variation**, which does not. Averaging by calendar month separates the first
one out.

In the altiplano the rainy season runs roughly from December to March, and the lake response
lags it by one to three months because the water has to run off and arrive. Look at *where*
the maximum falls, not only at how large the amplitude is: the lag between rainfall and lake
maximum is the catchment's response time.

In [ ]:
# ---------------------------------------------------------------------------
# Average water area by calendar month (usable months only)
# ---------------------------------------------------------------------------
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

cycle = good.groupby(good.index.month)["water_km2"].agg(["mean", "min", "max"])

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.fill_between(cycle.index, cycle["min"], cycle["max"], color=BLUE, alpha=0.18,
                label="min - max over the 10 years", zorder=2)
ax.plot(cycle.index, cycle["mean"], color=BLUE, linewidth=2.5,
        marker="o", markersize=6, markeredgecolor="white", markeredgewidth=1,
        label="monthly mean", zorder=3)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTHS)
ax.set_ylabel("Water area (km2)", fontsize=10, color=MUTED)
ax.set_title("Average seasonal cycle, 2012-2021", fontsize=11, color=INK, loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, labelcolor=INK)

fig.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.12)
plt.show()

print("Maximum on average :", MONTHS[cycle['mean'].idxmax() - 1])
print("Minimum on average :", MONTHS[cycle['mean'].idxmin() - 1])
print("Seasonal amplitude : {:,.0f} km2".format(cycle["mean"].max() - cycle["mean"].min()))

### Step 6 - Permanent and seasonal water, year by year

`YearlyHistory` classifies each pixel once per year with the band `waterClass`:

| Value | Class |
|---|---|
| 0 | no data |
| 1 | not water |
| 2 | **seasonal** water |
| 3 | **permanent** water |

This splits the total area into its two components. Total area can stay flat while permanent
water turns into seasonal water underneath - a system becoming less reliable without
becoming smaller. A stacked bar chart makes that visible; a single line does not.

In [ ]:
# ---------------------------------------------------------------------------
# Yearly classification: permanent (3) vs seasonal (2) water
# ---------------------------------------------------------------------------
yearly = (ee.ImageCollection("JRC/GSW1_4/YearlyHistory")
            .filterDate(START, END)
            .filterBounds(region))

def yearly_area(image):
    water_class = image.select("waterClass")

    seasonal_m2 = water_class.eq(2).multiply(pixel_area).rename("seasonal_m2")
    permanent_m2 = water_class.eq(3).multiply(pixel_area).rename("permanent_m2")

    stats = seasonal_m2.addBands(permanent_m2).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, scale=SCALE, maxPixels=1e10)

    return ee.Feature(None, {
        "year": image.date().get("year"),
        "seasonal_km2": ee.Number(stats.get("seasonal_m2")).divide(1e6),
        "permanent_km2": ee.Number(stats.get("permanent_m2")).divide(1e6),
    })

yearly_rows = [f["properties"] for f in yearly.map(yearly_area).getInfo()["features"]]

dfy = pd.DataFrame(yearly_rows).sort_values("year").set_index("year")
dfy["total_km2"] = dfy["permanent_km2"] + dfy["seasonal_km2"]
dfy

In [ ]:
# ---------------------------------------------------------------------------
# Stacked bars: how the total area is split between permanent and seasonal
# ---------------------------------------------------------------------------
DEEP, LIGHT = "#0a4a7a", "#8fc4e0"

fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(dfy.index, dfy["permanent_km2"], color=DEEP, label="permanent", zorder=3)
ax.bar(dfy.index, dfy["seasonal_km2"], bottom=dfy["permanent_km2"],
       color=LIGHT, label="seasonal", zorder=3)

ax.set_xticks(dfy.index)
ax.set_xticklabels(dfy.index, rotation=0)
ax.set_ylabel("Water area (km2)", fontsize=10, color=MUTED)
ax.set_title("Permanent and seasonal water per year", fontsize=11, color=INK, loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, labelcolor=INK, loc="lower left")

fig.subplots_adjust(left=0.08, right=0.98, top=0.91, bottom=0.10)
plt.show()

### Step 7 - Look at the two extreme years side by side

A number on a chart should always be traceable back to a map. We take the year with the
largest water area and the year with the smallest, and put them in a **split map**: one year
on the left of the slider, the other on the right, same view, same colours. Drag the handle
across the lake.

This is the moment to check that the minimum in the chart is a real retreat of the shoreline
and not an artefact - a cloudy year, a gap in the Landsat 7 stripes, or a polygon edge.

In [ ]:
# ---------------------------------------------------------------------------
# Split map: wettest year vs driest year
# ---------------------------------------------------------------------------
YEAR_MAX = int(dfy["total_km2"].idxmax())
YEAR_MIN = int(dfy["total_km2"].idxmin())

print("Largest water area :", YEAR_MAX, "-> {:,.0f} km2".format(dfy.loc[YEAR_MAX, "total_km2"]))
print("Smallest water area:", YEAR_MIN, "-> {:,.0f} km2".format(dfy.loc[YEAR_MIN, "total_km2"]))

vis_class = {"min": 0, "max": 3, "palette": ["#ffffff", "#ffffff", "#8fc4e0", "#0a4a7a"]}

image_max = ee.Image(yearly.filterDate(str(YEAR_MAX) + "-01-01",
                                       str(YEAR_MAX + 1) + "-01-01").first()).clip(region)
image_min = ee.Image(yearly.filterDate(str(YEAR_MIN) + "-01-01",
                                       str(YEAR_MIN + 1) + "-01-01").first()).clip(region)

Map_split = geemap.Map(center=[-16.35, -68.90], zoom=10)
Map_split.add_basemap("SATELLITE")
Map_split.split_map(
    left_layer=geemap.ee_tile_layer(image_max.select("waterClass").updateMask(
        image_max.select("waterClass").gt(1)), vis_class, str(YEAR_MAX)),
    right_layer=geemap.ee_tile_layer(image_min.select("waterClass").updateMask(
        image_min.select("waterClass").gt(1)), vis_class, str(YEAR_MIN)),
)
Map_split.centerObject(region, 10)

Map_split

### Step 8 - The whole record as an animation

The same yearly images, rendered as a GIF over the full 1984-2021 record. `visualize()`
turns the class values into RGB, and `getVideoThumbURL()` asks Earth Engine to assemble the
frames on the server and return a URL.

This is the single most effective way to present the result to a non-technical audience -
and it is also a diagnostic. Watch for frames that flicker white: those are years with poor
observation coverage, the same effect the bottom panel of the chart in step 4 quantified.

In [ ]:
# ---------------------------------------------------------------------------
# Animated GIF of the yearly water classification, 1984-2021
# ---------------------------------------------------------------------------
from IPython.display import Image as ShowImage

yearly_all = (ee.ImageCollection("JRC/GSW1_4/YearlyHistory")
                .filterBounds(region))

def to_rgb(image):
    return image.select("waterClass").visualize(min=0, max=3, palette=vis_class["palette"])

gif_params = {
    "dimensions": 520,
    "region": region,
    "framesPerSecond": 3,
    "crs": "EPSG:3857",
}

gif_url = yearly_all.map(to_rgb).getVideoThumbURL(gif_params)

print("Frames:", yearly_all.size().getInfo())
print(gif_url)

ShowImage(url=gif_url)   # may take up to a minute to appear

---

## 1.6 - Continuing the series after 2021 with Sentinel-2

The JRC product stops in December 2021. For anything more recent you have to produce the
classification yourself. We do it in the simplest defensible way: a monthly Sentinel-2
composite and **MNDWI** with a threshold at zero.

$$\text{MNDWI} = \frac{\rho_{GREEN} - \rho_{SWIR1}}{\rho_{GREEN} + \rho_{SWIR1}}
= \frac{B3 - B11}{B3 + B11}$$

MNDWI (Xu, 2006) is preferred over NDWI here because it replaces the near infrared with the
SWIR, which suppresses the confusion between water and built-up surfaces.

We start in **2019**, three years before the JRC record ends, so that the two curves overlap
and can be compared instead of merely joined.

> **These two curves will not coincide, and you must say so when you present them.**
> Different sensor (Sentinel-2 at 10-20 m vs Landsat at 30 m), different classifier (a fixed
> MNDWI threshold vs the JRC expert system), different cloud handling. The overlap period is
> what tells you the size of the offset. Reporting a jump at the junction of two methods as
> if it were a change in the lake is one of the most common errors in this kind of analysis.

In [ ]:
# ---------------------------------------------------------------------------
# Sentinel-2 Level-2A, cloud-masked, over the region
# ---------------------------------------------------------------------------
import datetime

S2_START = "2019-01-01"
# First day of the current month, so the series always runs up to today
S2_END = datetime.date.today().replace(day=1).isoformat()

def mask_s2(image):
    # SCL classes to discard: 3 shadow, 8-9 cloud, 10 cirrus, 11 snow
    scl = image.select("SCL")
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                .And(scl.neq(10)).And(scl.neq(11)))

    optical = image.select(["B3", "B11"]).multiply(0.0001).updateMask(clear)

    # multiply() drops the metadata, and we need the date to build monthly
    # composites below - so it has to be copied back explicitly.
    return ee.Image(optical.copyProperties(image, ["system:time_start"]))

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(S2_START, S2_END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .map(mask_s2))

print("Period :", S2_START, "to", S2_END)
print("Images :", s2.size().getInfo())

### One composite per month

`ee.List.sequence` gives us the month numbers 0, 1, 2, ... and `advance()` turns each of
them into a start date. For every month we take the **median** of the cloud-masked images -
the median is robust to the few cloud pixels the SCL mask lets through - and compute MNDWI.

`n_images` records how many scenes went into each composite, so that months with no
acquisitions can be dropped before anything is computed on them.

In [ ]:
# ---------------------------------------------------------------------------
# Monthly MNDWI composites
# ---------------------------------------------------------------------------
n_months = ee.Date(S2_END).difference(ee.Date(S2_START), "month").round()
print("Months to build:", n_months.getInfo())
offsets = ee.List.sequence(0, n_months.subtract(1))

def monthly_mndwi(offset):
    start = ee.Date(S2_START).advance(offset, "month")
    month_images = s2.filterDate(start, start.advance(1, "month"))

    mndwi = month_images.median().normalizedDifference(["B3", "B11"]).rename("MNDWI")

    return mndwi.set("date", start.format("YYYY-MM-dd")).set("n_images", month_images.size())

# Months without any image are removed before anything is computed on them
mndwi_monthly = (ee.ImageCollection(offsets.map(monthly_mndwi))
                   .filter(ee.Filter.gt("n_images", 0)))

print("Months with data:", mndwi_monthly.size().getInfo())

### The same area recipe, applied to the MNDWI mask

Exactly the four steps of section 1.4, with `MNDWI > 0` as the mask instead of
`water == 2`, and the same coverage check as in section 1.5. The role that the JRC no-data
value played there is played here by the **cloud mask**: a pixel hidden by cloud is not land,
it is unobserved, and it has to be counted as such.

`image.mask()` returns the mask itself as a normal image, which is how we measure the
observed area. `unmask(0)` then turns any remaining masked pixel into a zero, so the sum is
always a number and never `null`.

The scale is 60 m, the same as the JRC series, so the two numbers are computed on
comparable grids.

In [ ]:
# ---------------------------------------------------------------------------
# Water area from Sentinel-2: MNDWI > 0, with the same coverage check
# ---------------------------------------------------------------------------
def s2_water_area(image):
    mndwi = image.select("MNDWI")

    water_m2 = mndwi.gt(0).unmask(0).multiply(pixel_area).rename("water_m2")
    valid_m2 = mndwi.mask().gt(0).unmask(0).multiply(pixel_area).rename("valid_m2")

    stats = water_m2.addBands(valid_m2).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, scale=SCALE, maxPixels=1e10)

    return ee.Feature(None, {
        "date": image.get("date"),
        "n_images": image.get("n_images"),
        "water_km2": ee.Number(stats.get("water_m2")).divide(1e6),
        "valid_km2": ee.Number(stats.get("valid_m2")).divide(1e6),
    })

# This may take 2-3 minutes: about 90 monthly composites are built on the server
s2_rows = [f["properties"] for f in mndwi_monthly.map(s2_water_area).getInfo()["features"]]

df_s2 = pd.DataFrame(s2_rows)
df_s2["date"] = pd.to_datetime(df_s2["date"])
df_s2 = df_s2.sort_values("date").set_index("date")

df_s2["coverage"] = df_s2["valid_km2"] / region_km2
df_s2["usable"] = df_s2["coverage"] >= MIN_COVERAGE

good_s2 = df_s2[df_s2["usable"]]

print("Monthly composites  :", len(df_s2))
print("Usable months       :", len(good_s2))
df_s2.head()

In [ ]:
# ---------------------------------------------------------------------------
# The two series on one chart: JRC (to 2021) and Sentinel-2 (from 2019)
# ---------------------------------------------------------------------------
ORANGE = "#e07b39"
OVERLAP_START, OVERLAP_END = "2019-01-01", "2022-01-01"

fig, ax = plt.subplots(figsize=(12, 5.5))

# Shade the overlap period, where the two methods can be compared directly
ax.axvspan(pd.Timestamp(OVERLAP_START), pd.Timestamp(OVERLAP_END),
           color=GRID, alpha=0.45, zorder=1)

ax.plot(good.index, good["water_km2"], color=BLUE, linewidth=2,
        label="JRC monthly history (Landsat)", zorder=3)
ax.plot(good_s2.index, good_s2["water_km2"], color=ORANGE, linewidth=2,
        label="Sentinel-2, MNDWI > 0", zorder=3)

ax.text(pd.Timestamp("2020-07-01"), ax.get_ylim()[1], " overlap ",
        fontsize=9, color=MUTED, va="top", ha="center")

ax.set_ylabel("Water area (km2)", fontsize=10, color=MUTED)
ax.set_title("Surface water area: JRC record extended with Sentinel-2",
             fontsize=11, color=INK, loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, labelcolor=INK, loc="lower left")

fig.subplots_adjust(left=0.08, right=0.98, top=0.91, bottom=0.10)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# How large is the offset between the two methods, over the overlap period?
# ---------------------------------------------------------------------------
a = good["water_km2"].loc[OVERLAP_START:OVERLAP_END].resample("MS").mean()
b = good_s2["water_km2"].loc[OVERLAP_START:OVERLAP_END].resample("MS").mean()

both = pd.DataFrame({"jrc": a, "s2": b}).dropna()
both["difference"] = both["s2"] - both["jrc"]

print("Months compared      :", len(both))
print("Mean JRC area        : {:8,.1f} km2".format(both["jrc"].mean()))
print("Mean Sentinel-2 area : {:8,.1f} km2".format(both["s2"].mean()))
print("Mean difference      : {:+8,.1f} km2  ({:+.1f}%)".format(
    both["difference"].mean(), 100 * both["difference"].mean() / both["jrc"].mean()))
print("Correlation          : {:8.3f}".format(both["jrc"].corr(both["s2"])))

### Reading the comparison

Two statistics, two different meanings:

- The **mean difference** is a systematic offset: one method consistently finds more water
  than the other. An offset can be corrected, or simply reported, and it does not prevent
  you from reading the trend.
- The **correlation** says whether the two methods see the same *movements*. A high
  correlation with a large offset means the extended series is usable for change detection
  but not for absolute area. A low correlation means the two methods are responding to
  different things and the series cannot be joined at all.

Where does a difference come from? Sentinel-2 resolves narrow features that Landsat misses,
which adds area. A fixed MNDWI threshold of 0 classifies shallow turbid water and wet soil
differently from the JRC expert system, which can add or remove area. The median composite
of a month is not the same object as a monthly classification. All three act at once.

---

## 1.7 - Ejercicio / Exercise

**Consigna / Task**

Choose **two contrasting water bodies** inside the study area and compare their behaviour.
Suggested pairs:

- Lago Menor (Wi&ntilde;aymarka, shallow) against a bay of Lago Grande (Chucuito, deep)
- A stretch of the Desaguadero floodplain against the lake it drains
- Any two reservoirs or wetlands you can identify on the occurrence layer

For each of the two:

1. Draw the polygon and re-run sections 1.5 and 1.6, saving the results under different
   names (`df_a`, `dfy_a`, then `df_b`, `dfy_b`).
2. Compute, for each one:
   - the mean water area over 2012-2021,
   - the seasonal amplitude (mean of the wettest month minus mean of the driest month),
   - the **relative** amplitude, that is the amplitude divided by the mean area,
   - the share of the area that was permanent in 2012 and in 2021.
3. Put the two series on a single chart. Because the absolute areas differ, plot them as a
   **percentage of each body's own mean area**, otherwise the larger one dominates the axis.

**Answer in writing**

- Which of the two has the larger *relative* seasonal amplitude, and what does that say about
  its depth and the shape of its bed?
- Do the two reach their maximum in the same month? If not, what could explain the lag -
  catchment size, snowmelt, upstream regulation?
- Did the permanent fraction change between 2012 and 2021 in either one? Note that a
  decrease in permanent water with a stable total is a loss of reliability, not a loss of
  water.
- List the sources of uncertainty that affect your comparison, and state which of your
  conclusions survive them and which do not.

**Entregable / Deliverable**

The comparison chart, the table of statistics, and 15-20 lines of interpretation that make
explicit the distinction between *surface area* and *water volume*.

---

## Referencias / References

**The product**

- Pekel, J.-F., Cottam, A., Gorelick, N., Belward, A.S. (2016). High-resolution mapping of
  global surface water and its long-term changes. *Nature*, 540, 418-422.
  <https://doi.org/10.1038/nature20584>
- Global Surface Water Explorer (interactive viewer): <https://global-surface-water.appspot.com/>
- Data and documentation (JRC): <https://global-surface-water.appspot.com/download>

**Earth Engine datasets**

- Mapping layers v1.4: <https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater>
- Monthly history v1.4: <https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_MonthlyHistory>
- Yearly history v1.4: <https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_YearlyHistory>
- Monthly recurrence v1.4: <https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_MonthlyRecurrence>
- Sentinel-2 SR Harmonized: <https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED>

**Indices and methods**

- Xu, H. (2006). Modification of normalised difference water index (NDWI) to enhance open
  water features in remotely sensed imagery. *International Journal of Remote Sensing*,
  27(14), 3025-3033. (MNDWI)
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI) in the
  delineation of open water features. *International Journal of Remote Sensing*, 17(7),
  1425-1432.

**Tools**

- Earth Engine guide - reducing regions: <https://developers.google.com/earth-engine/guides/reducers_reduce_region>
- Earth Engine guide - scale: <https://developers.google.com/earth-engine/guides/scale>
- geemap documentation: <https://geemap.org>

**Context: the TDPS system**

- Revollo, M.M. (2001). Management issues in the Lake Titicaca and Lake Poopo system.
  *Lakes & Reservoirs: Research and Management*, 6(3), 199-207.
- Satge, F. et al. (2017). Role of climate variability and human activity on Poopo Lake
  droughts between 1990 and 2015 assessed using remote sensing data. *Remote Sensing*,
  9(3), 218.